### Data Ingesion

In [2]:
### Document Data Structure

from langchain_core.documents import Document

In [3]:
doc = Document(
    page_content= "This is the page content of RAG",
    metadata = {
        "source":"example.txt",
        "pages":1,
        "author": "Ayan Guin",
        "date_created" : "2026-01-01"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Ayan Guin', 'date_created': '2026-01-01'}, page_content='This is the page content of RAG')

In [4]:
## Create a simple txt file

import os
os.makedirs("../data/txt_files",exist_ok=True)

In [5]:
sample_texts={
    "../data/txt_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/txt_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

In [6]:
for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print(" 👉🏻 sample file created!")

 👉🏻 sample file created!


In [8]:
### TextLoader

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/txt_files/python_intro.txt", encoding="utf-8")
document = loader.load()
document

[Document(metadata={'source': '../data/txt_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]

In [9]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## Load all the text files from the directory
dir_loader = DirectoryLoader(
    "../data/txt_files",
    glob="**/*.txt", #pattern to match file
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=True
)

documents=dir_loader.load()
documents

100%|██████████| 2/2 [00:00<00:00, 1546.86it/s]


[Document(metadata={'source': '../data/txt_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.'),
 Document(metadata={'source': '../data/txt_files/machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised 

In [13]:
### PDF Loader
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader

## Load all pdf from directory
pdf_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", #pattern to match file
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

pdf_documents=pdf_loader.load()
pdf_documents

100%|██████████| 2/2 [00:00<00:00, 29.91it/s]


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-01-19T14:13:57+00:00', 'source': '../data/pdf/ML4NLU_Flipped_Class_QA.pdf', 'file_path': '../data/pdf/ML4NLU_Flipped_Class_QA.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-01-19T14:13:57+00:00', 'trapped': '', 'modDate': "D:20260119141357+00'00'", 'creationDate': "D:20260119141357+00'00'", 'page': 0}, page_content='Machine Learning for Natural Language Understanding\nLecture 3 – Neural Networks\nQ: Why is the quadratic loss not ideal for binary classification?\nA: Quadratic loss penalizes errors symmetrically and leads to slow learning and poor probabilistic\ninterpretation for binary outputs. Logistic loss is better suited.\nQ: How is the output of an artificial neuron calculated?\nA: By computing a weighted sum of inputs plus bias and applying an activ

### embedding and vectorStoreDB

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid # this is to store the datas with unique IDs.
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()